# L5-1 起报时次与时效

四个容易混的概念：

| 概念 | 含义 | 例子 |
|---|---|---|
| 起报时间 | 模式从哪个时刻开始算，ECMWF 只有 00、12 UTC | 2025061400（北京时 6 月 14 日 08 时） |
| 资料到达时间 | 产品什么时候能拿到，通常比起报晚几个小时 | 00 UTC 的资料北京时 7 点后才齐 |
| 预报时效 | 从起报时间往后多少小时 | 30 小时 |
| 有效时间 | 起报时间 + 时效，写稿时要换成北京时 | 6 月 15 日 14 时（北京时） |

In [1]:
# 改了 common/、weather_tools/ 里的代码后自动重新加载，不用重启内核
%load_ext autoreload
%autoreload 2

from datetime import datetime

from common.clock import BJ_TZ, now_bj
from weather_tools import timeutil

print("课程当前北京时：", now_bj())

课程当前北京时： 2026-10-29 10:31:55.027121+08:00


## 1. 换算小测

先在纸上算，再运行核对：北京时 2025-06-15 14 时，

- 对应 2025061400 起报的哪个时效？
- 对应 2025061312 起报的哪个时效？

In [2]:
print(timeutil.lead_time_for("2025061400", "2025-06-15 14:00"))
print(timeutil.lead_time_for("2025061312", "2025-06-15 14:00"))

30
42


## 2. “现在该用哪个起报时次”

规则来自 agent_dev `src/tools/ToolBase.py:64-73`（北京时，只有 08、20 两个时次）：

| 北京时 | 用哪个起报时次 | 对应 UTC |
|---|---|---|
| 06:00～13:59 | 当天 08 时 | 当天 00 |
| 14:00～23:59 | 当天 20 时 | 当天 12 |
| 00:00～05:59 | 前一天 20 时 | 前一天 12 |

这是“资料大约几点到”的业务经验，写成了代码。

In [4]:
for s in ["2026-10-26 05:59", "2026-10-26 06:00", "2026-10-26 13:59", "2026-10-26 14:00", "2026-10-26 23:59"]:
    dt = datetime.strptime(s, "%Y-%m-%d %H:%M").replace(tzinfo=BJ_TZ)
    print(s, "→", timeutil.latest_init_time(dt))

2026-10-26 05:59 → 2026102512
2026-10-26 06:00 → 2026102600
2026-10-26 13:59 → 2026102600
2026-10-26 14:00 → 2026102612
2026-10-26 23:59 → 2026102612


## 3. 一条规则，逐时过一遍

0～23 时全跑一遍，确认切换点只落在 06 时和 14 时两处；代码和上表对不上就是改坏了。

In [5]:
def rule_label(hour: int) -> str:
    """把 ToolBase 规则翻译成人话，用来和代码结果对照。"""
    if hour >= 14:
        return "当天20时"
    if hour >= 6:
        return "当天08时"
    return "前一天20时"


for h in range(24):
    dt = datetime(2025, 6, 14, h, 0, tzinfo=BJ_TZ)
    print(f"{h:02d} 时：{rule_label(h)} → {timeutil.latest_init_time(dt)} UTC")

00 时：前一天20时 → 2025061312 UTC
01 时：前一天20时 → 2025061312 UTC
02 时：前一天20时 → 2025061312 UTC
03 时：前一天20时 → 2025061312 UTC
04 时：前一天20时 → 2025061312 UTC
05 时：前一天20时 → 2025061312 UTC
06 时：当天08时 → 2025061400 UTC
07 时：当天08时 → 2025061400 UTC
08 时：当天08时 → 2025061400 UTC
09 时：当天08时 → 2025061400 UTC
10 时：当天08时 → 2025061400 UTC
11 时：当天08时 → 2025061400 UTC
12 时：当天08时 → 2025061400 UTC
13 时：当天08时 → 2025061400 UTC
14 时：当天20时 → 2025061412 UTC
15 时：当天20时 → 2025061412 UTC
16 时：当天20时 → 2025061412 UTC
17 时：当天20时 → 2025061412 UTC
18 时：当天20时 → 2025061412 UTC
19 时：当天20时 → 2025061412 UTC
20 时：当天20时 → 2025061412 UTC
21 时：当天20时 → 2025061412 UTC
22 时：当天20时 → 2025061412 UTC
23 时：当天20时 → 2025061412 UTC


边界提早了，意味着 06:00 就假定当天 08 时次的资料已到。真遇到资料迟到，改的是这两个边界值，
不是另起一套规则——两三处各写一套，才是这类 bug 的根源。